# DQA Figure Generation

## Setup

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
if ROOT.name == "phase 4 - report":
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "output" / "4-cleaning pipeline" / "Global_Scorecard_Comparison.csv"
OUTPUT_DIR = ROOT / "phase 4 - report" / "images" / "dqa"

TABLE_ORDER = [
    "Flights",
    "Weather_Observations",
    "Carriers",
    "Aircrafts",
    "Stations",
    "Active_Weather",
    "Cancellation",
]

DIMENSION_ORDER = [
    "Completeness",
    "Uniqueness",
    "Validity",
    "Consistency",
    "Timeliness",
]

SCORE_COLORS = LinearSegmentedColormap.from_list(
    "score_gradient",
    ["#c43c39", "#f2c14e", "#4c956c"],
)
DELTA_COLORS = LinearSegmentedColormap.from_list(
    "delta_gradient",
    ["#f7f7f7", "#b8d8f0", "#4a90c2"],
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(INPUT_PATH)

## Helper Functions

In [ ]:
def prepare_matrix(df: pd.DataFrame, value_column: str) -> pd.DataFrame:
    matrix = df.pivot_table(
        index="Table",
        columns="Dimension",
        values=value_column,
        aggfunc="first",
    )
    matrix = matrix.reindex([table for table in TABLE_ORDER if table in matrix.index])
    matrix = matrix.reindex(columns=[dim for dim in DIMENSION_ORDER if dim in matrix.columns])
    return matrix

In [ ]:
def delta_value(value: float) -> float:
    if pd.isna(value):
        return float("nan")
    return min(max(value, 0), 1)

In [ ]:
def format_value(value: float, signed: bool = False) -> str:
    if pd.isna(value):
        return ""
    if value in (0, 1):
        return f"{value:+.0f}" if signed else f"{value:.0f}"
    return f"{value:+.4f}" if signed else f"{value:.4f}"

In [ ]:
def draw_matrix(
    values: pd.DataFrame,
    color_values: pd.DataFrame,
    title: str,
    output_name: str,
    cmap: LinearSegmentedColormap,
    signed: bool = False,
    show_colorbar: bool = False,
    colorbar_label: str = "",
) -> None:
    fig, ax = plt.subplots(figsize=(12.8, 6.2))
    image = ax.imshow(color_values.to_numpy(dtype=float), cmap=cmap, vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(values.columns)))
    wrapped_columns = [label.replace("_", "\n") if len(label) > 11 else label for label in values.columns]
    wrapped_index = [label.replace("_", "\n") if len(label) > 12 else label for label in values.index]
    ax.set_xticklabels(wrapped_columns, rotation=0, ha="center", fontsize=10)
    ax.set_yticks(range(len(values.index)))
    ax.set_yticklabels(wrapped_index, rotation=0, fontsize=10)

    ax.set_xlabel("Quality dimension", labelpad=14)
    ax.set_ylabel("")
    ax.text(-0.08, 1.005, "Table", transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold", color="#222222")
    ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=24)

    ax.set_xticks([x - 0.5 for x in range(1, len(values.columns))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(values.index))], minor=True)
    ax.grid(which="minor", color="white", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)

    for row_idx, table in enumerate(values.index):
        for col_idx, dimension in enumerate(values.columns):
            value = values.loc[table, dimension]
            if pd.isna(value):
                continue
            text = format_value(value, signed=signed)
            color_value = color_values.loc[table, dimension]
            if signed:
                text_color = "#222222" if color_value < 0.30 else "white"
            else:
                text_color = "white" if color_value <= 0.35 or color_value >= 0.75 else "#222222"
            ax.text(col_idx, row_idx, text, ha="center", va="center", color=text_color, fontsize=10)

    for spine in ax.spines.values():
        spine.set_visible(False)

    if show_colorbar:
        colorbar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.035)
        colorbar.ax.set_title(colorbar_label, fontsize=9, pad=8)
        colorbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

    fig.tight_layout(pad=1.5)
    fig.savefig(OUTPUT_DIR / output_name, dpi=220, bbox_inches="tight")
    plt.close(fig)

## Generate Scorecard Plots

In [ ]:
before = prepare_matrix(df, "Score_before")
after = prepare_matrix(df, "Score_after")
delta = prepare_matrix(df, "Delta")

draw_matrix(
    values=before,
    color_values=before,
    title="DQA Scorecard Before Cleaning",
    output_name="dqa_scorecard_before.png",
    cmap=SCORE_COLORS,
    show_colorbar=True,
    colorbar_label="Quality score",
)
draw_matrix(
    values=after,
    color_values=after,
    title="DQA Scorecard After Cleaning",
    output_name="dqa_scorecard_after.png",
    cmap=SCORE_COLORS,
    show_colorbar=True,
    colorbar_label="Quality score",
)
draw_matrix(
    values=delta,
    color_values=delta.map(delta_value),
    title="DQA Score Improvement After Cleaning",
    output_name="dqa_scorecard_delta.png",
    cmap=DELTA_COLORS,
    signed=True,
    show_colorbar=True,
    colorbar_label="Score improvement (after - before)",
)

sorted(path.name for path in OUTPUT_DIR.glob("dqa_scorecard*.png"))